In [1]:
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

import pandas as pd

import ipywidgets as widgets

from IPython.display import (
    display,
    clear_output
)

from src.data_io import (
    load_dataframe,
    save_dataframe
)

In [2]:
economy_df = load_dataframe(
    "../data/interim/economy_news_sentiment_input.parquet"
)

In [3]:
sample_news = (
    economy_df["title"]
    .sample(
        20,
        random_state=42
    )
)

In [4]:
## Data Initialization
news_title = sample_news.to_list()
label_result = []
current_index = 0

# Element UI
output_text = widgets.Output()
text_input = widgets.Text(
    placeholder="Type: Positive / Neutral / Negative",
    description="Label:"
)
button_submit = widgets.Button(description="Save & Next")

def news_display():
    """
    Updates and display the current news headline in the output widget.

    
    This function manages the interactive UI states. If there are remaining headlines, 
    it prints the current progress and the headline text. Once all headlines 
    are labeled, it disables the input fields and automatically saves the 
    collected labels into a global pandas DataFrame named 'df_final'.

    Parameters:
    -----------
        None

    Returns:
    --------
        None
    """
    with output_text:
        clear_output()
        if current_index < len(news_title):
            print(f"News {current_index + 1} from {len(news_title)}")
            print("-" * 50)
            print(news_title[current_index])
            print("-" * 50)
        else:
            print("All news already labeled!")
            text_input.disabled=True
            button_submit.disabled=True
            # Create dataframe automatically when its done
            global df_final
            df_final = pd.DataFrame(
                {
                    'Title': news_title,
                    'Label': label_result
                }
            )
            print("\nDataFrame 'df_final' successfully created. Type 'df_final' in a new cell to check it out!")

def on_button_click(b):
    """
     Handles the submit button click event to process and store the user's input.
    
    This function retrieves the text from the text input widget, standardizes its 
    capitalization, and appends it to the global label list. It then resets 
    the input field and increments the current index to load the next headline.
    
    Parameters:
    ----------
        b : ipywidgets.Button
            The button instance that triggered the click event.

    Returns:
        None
    """
    global current_index
    if text_input.value.strip() != "":
        label_result.append(text_input.value.strip().capitalize())
        text_input.value = ""
        current_index += 1
        news_display()

button_submit.on_click(on_button_click)

# display widget
display(output_text, widgets.HBox([text_input, button_submit]))
news_display()

Output()

In [6]:
df_final

,Title,Label
0,"BCA Syariah Cetak Laba Bersih Rp 117,6 Miliar ...",Positive
1,"SVB Kebangkrutan Bank Terbesar Sejak 2008, Mil...",Negative
2,"AS Pening, 'Badai' Baru Bikin Penampakan Reses...",Negative
3,"Pertumbuhan Ekonomi Indonesia Ditaksir 4,9 sam...",Positive
4,Jokowi Buka Opsi Impor Beras 500 Ribu Ton Lagi,Neutral
5,Daftar Harga Emas Antam Hari Ini Usai Turun Rp...,Neutral
6,"Turun Ceban, Harga Emas Antam Hari Ini Dijual ...",Neutral
7,Entitas Nikel Harita Group Targetkan Pendapata...,Positive
8,"Ramai Panik Silicon Valley Bank Bangkrut, Bos ...",Negative
9,"Lakukan Transisi Energi, PGE Bukukan Pendapata...",Positive


In [ ]:
save_dataframe(
    df_final,
    "../data/interim/manual_sentiment_labels.parquet"
)